# MLP / Feed-Forward Layer Analysis — 4 models

Third companion to `embedding_analysis.ipynb` and `attention_analysis.ipynb`,
working through `weights_analysis/todo_mlp.txt` for four checkpoints:

| Key | Directory | Role |
|-----|-----------|------|
| `Llama-3-8B`        | `llama-3-8b`                   | original Meta base model |
| `SinLlama_v01`      | `SinLlama_v01`                | Sinhala model from Llama-3 (CPT) — **parent of cpt & instruct** |
| `SinLlama_cpt`      | `SinLlama_cpt`                 | branched from v01 |
| `SinLlama_Instruct` | `SinLlama_Backtrianx_Instruct`| branched from v01 (Bactrian-X instruct) |

**Lineage:** `Llama-3-8B → SinLlama_v01 → {SinLlama_cpt, SinLlama_Instruct}` —
`cpt` and `instruct` are **parallel branches off v01**, not sequential to each
other. Comparisons follow this order.

Each layer's MLP is a **SwiGLU** block
`down_proj( SiLU(gate_proj(x)) ⊙ up_proj(x) )`:

* `gate_proj.weight`, `up_proj.weight` : `[intermediate, hidden] = [14336, 4096]`
* `down_proj.weight` : `[hidden, intermediate] = [4096, 14336]`

A **neuron** = one of the 14,336 intermediate channels: it *reads* the residual
through its `gate`/`up` rows and *writes* back through its `down` column. All
four models share the identical MLP architecture (only the embedding / LM-head
vocab differs), so every neuron is comparable per `(layer, index)` and we can
watch how each training stage reshaped the feed-forward computation.

### Two regimes
* **Part A — static / weight-space** (§1-4, 9-11): reads only `gate/up/down`
  weights via `safetensors`; no full model instantiation. Runs anywhere.
* **Part B — dynamic / activation-space** (§5-8, 12-17): needs real forward
  passes with **hooks on the MLP sub-modules** to capture gate/up/post-SwiGLU
  activations. Loads each full 8B model one at a time (~16 GB) and frees it.
  Auto-selects GPU if it fits, else CPU.

> **Memory:** one model's `gate/up/down` is ≈ 22 GB in float32 — holding all
> four at once (~90 GB) is impractical, so Part A streams **layer-by-layer**
> (base + one variant per layer, peak ≈ 1.5 GB), reducing each to compact
> summaries. Part B loads one full model at a time. Comfortable on the MI300X
> pod; trim `MODEL_DIRS` / `DYN_MODELS` on smaller boxes.

## 0. Setup — imports, configuration, helpers

In [ ]:
import os, re, json, math, gc, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from safetensors import safe_open
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram
import umap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# --- Portable model paths (laptop AND pod; case-insensitive dir match) ----
CANDIDATE_ROOTS = [
    "/ml/SinLlama_CPT/models",
    os.path.expanduser("~/sinllama-continual-pretraining/models"),
    "/root/sinllama-continual-pretraining/models",
    "./models",
]
MODELS_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isdir(r)), CANDIDATE_ROOTS[0])
# Order follows the model lineage: llama -> v01 -> {cpt, instruct}.
MODEL_DIRS = {
    "Llama-3-8B":        ["llama-3-8b"],
    "SinLlama_v01":      ["SinLlama_v01"],
    "SinLlama_cpt":      ["SinLlama_cpt"],
    "SinLlama_Instruct": ["SinLlama_Backtrianx_Instruct", "SinLlama_Backtrianx_instruct"],
}
def resolve_dir(root, candidates):
    "Case-insensitive match of a model directory under `root`."
    existing = {d.lower(): d for d in os.listdir(root)
                if os.path.isdir(os.path.join(root, d))}
    for c in candidates:
        if c.lower() in existing:
            return os.path.join(root, existing[c.lower()])
    raise FileNotFoundError(f"none of {candidates} under {root}")
MODEL_PATHS = {k: resolve_dir(MODELS_ROOT, v) for k, v in MODEL_DIRS.items()}
MODEL_KEYS = list(MODEL_PATHS)          # canonical iteration order
REF = MODEL_KEYS[0]                     # base model for single-model reference plots

# Architecture constants (identical for every model; verified in §1).
N_LAYERS, HIDDEN, INTERMEDIATE = 32, 4096, 14336

FIG_DIR = os.path.join(os.path.dirname(MODELS_ROOT.rstrip("/")),
                       "weights_analysis", "figures_mlp")
if not os.path.isdir(os.path.dirname(FIG_DIR)):
    FIG_DIR = "figures_mlp"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, name), bbox_inches="tight"); plt.show()

DEPTH_CMAP = plt.cm.viridis

# --- Spectral helpers (Part A) --------------------------------------------
# gate/up are [14336, 4096] and down is [4096, 14336]; the min dimension is 4096
# for all three, so the non-zero singular values come from the smaller 4096x4096
# Gram matrix (W^T W or W W^T) far more cheaply than a full SVD of the tall/wide
# matrix — one eigvalsh of a 4096^2 matrix instead of gesdd on 14336x4096.
def sing_vals(mat):
    "Singular values (descending) via the smaller Gram matrix."
    G = mat.T @ mat if mat.shape[1] <= mat.shape[0] else mat @ mat.T
    ev = np.linalg.eigvalsh(G)[::-1]
    return np.sqrt(np.clip(ev, 0.0, None))

def rank_metrics(sv):
    "Effective (exp-entropy), stable and numerical rank from singular values."
    sv = sv[sv > 0]
    p = sv / sv.sum()
    eff = float(np.exp(-(p * np.log(p)).sum()))
    stable = float((sv ** 2).sum() / (sv[0] ** 2))
    numer = int((sv > sv[0] * 1e-3).sum())
    return eff, stable, numer

print("MODELS_ROOT:", MODELS_ROOT, "| figures ->", FIG_DIR)
for k, p in MODEL_PATHS.items():
    print(f"  {k:20s} -> {p}")

# PART A — Static / weight-space analysis

Reads only the `gate/up/down` projection matrices; no full model instantiation.
Everything below is driven by compact per-`(layer, proj)` and per-neuron
summaries built in §1, so the multi-GB weight tensors are never held for more
than one layer at a time.

## A1 · §1 Load & inspect — dimensions, dtype, parameter budget

Verify the MLP architecture from each `config.json`, confirm the stored dtype,
and report parameters per MLP layer, total MLP parameters and their share of the
model (attention + MLP + tied-free embeddings).

In [ ]:
def model_dims(path):
    cfg = json.load(open(os.path.join(path, "config.json")))
    return (cfg["hidden_size"], cfg["intermediate_size"], cfg["num_hidden_layers"],
            cfg["vocab_size"], cfg["num_attention_heads"], cfg["num_key_value_heads"],
            cfg.get("torch_dtype"), cfg.get("hidden_act"))

def _shard_of(path, name):
    return os.path.join(path, json.load(open(os.path.join(
        path, "model.safetensors.index.json")))["weight_map"][name])

rows = []
for key in MODEL_KEYS:
    H, I, Ln, V, Hd, KV, td, act = model_dims(MODEL_PATHS[key])
    assert (H, I, Ln) == (HIDDEN, INTERMEDIATE, N_LAYERS), f"{key}: unexpected dims"
    kv_dim   = H * KV // Hd
    mlp_per  = 3 * I * H
    mlp_tot  = mlp_per * Ln
    attn_tot = Ln * (2 * H * H + 2 * H * kv_dim)
    embed    = 2 * V * H                      # input embeddings + untied lm_head
    total    = mlp_tot + attn_tot + embed
    with safe_open(_shard_of(MODEL_PATHS[key], "model.layers.0.mlp.gate_proj.weight"),
                   framework="pt", device="cpu") as f:
        dtype = f.get_tensor("model.layers.0.mlp.gate_proj.weight").dtype
    rows.append({"model": key, "act": act, "dtype": str(dtype),
                 "expansion": round(I / H, 3), "params/MLP": f"{mlp_per:,}",
                 "MLP params": f"{mlp_tot/1e9:.2f}B", "MLP %": round(100*mlp_tot/total, 1)})
display(pd.DataFrame(rows).set_index("model"))
print(f"gate/up = [{INTERMEDIATE}, {HIDDEN}], down = [{HIDDEN}, {INTERMEDIATE}] "
      f"| SwiGLU expansion = {INTERMEDIATE/HIDDEN:.1f}x")

SUMMARY = {k: {} for k in MODEL_KEYS}          # scalar metrics for the final table

### A1 · reduction pass (weight-space)

The one heavy pass: stream **layer-by-layer** (base + one variant per layer),
computing every weight-space summary used downstream — per-`(layer, proj)`
statistics & norms, singular values (effective / stable / numerical rank),
per-neuron down-column norms, a sub-sample of neurons for §10-11, a weight-value
sample for §2, and per-layer cosine drift from base — then free the tensors.
Peak RAM ≈ 1.5 GB. Expect a few minutes (96 Gram-based spectra).

In [ ]:
def load_layer_proj(path, L):
    "Load gate/up/down weight for one decoder layer as float32 numpy + dtype."
    idx = json.load(open(os.path.join(path, "model.safetensors.index.json")))["weight_map"]
    out, dtype = {}, None
    for p in ("gate", "up", "down"):
        name = f"model.layers.{L}.mlp.{p}_proj.weight"
        with safe_open(os.path.join(path, idx[name]), framework="pt", device="cpu") as f:
            t = f.get_tensor(name)
        out[p] = t.to(torch.float32).numpy(); dtype = t.dtype
    return out, dtype

NSAMP_PER_LAYER = 160                 # neurons sampled / layer for §10-11
WSAMPLE_PER_LP  = 1200                # weight values sampled / (layer, proj) for §2
SPEC_LAYERS     = [0, N_LAYERS // 2, N_LAYERS - 1]
PROJS           = ("gate", "up", "down")

STATREC   = []                                             # per (model, layer, proj)
SVEFF     = {k: {p: [] for p in PROJS} for k in MODEL_KEYS} # effective rank / layer
SVSTAB    = {k: {p: [] for p in PROJS} for k in MODEL_KEYS} # stable rank / layer
SVSPEC    = {}                                             # (key, L, proj) -> sv
DRIFT     = {k: {p: [] for p in PROJS} for k in MODEL_KEYS} # cosine to base / layer
WSAMPLE   = {k: {p: [] for p in PROJS} for k in MODEL_KEYS}
NEUR      = {}                                             # key -> (feats, layer_of)
DOWN_NORM = {k: np.zeros((N_LAYERS, INTERMEDIATE), np.float32) for k in MODEL_KEYS}
_nf = {k: [] for k in MODEL_KEYS}; _nl = {k: [] for k in MODEL_KEYS}

t0 = time.time()
for L in range(N_LAYERS):
    baseW, _ = load_layer_proj(MODEL_PATHS[REF], L)
    for key in MODEL_KEYS:
        W = baseW if key == REF else load_layer_proj(MODEL_PATHS[key], L)[0]
        for p in PROJS:
            m = W[p]
            sv = sing_vals(m); eff, stab, numer = rank_metrics(sv)
            SVEFF[key][p].append(eff); SVSTAB[key][p].append(stab)
            if L in SPEC_LAYERS:
                SVSPEC[(key, L, p)] = sv
            STATREC.append({"model": key, "layer": L, "proj": p,
                "mean": float(m.mean()), "std": float(m.std()),
                "min": float(m.min()), "max": float(m.max()),
                "median": float(np.median(m)),
                "p1": float(np.percentile(m, 1)), "p99": float(np.percentile(m, 99)),
                "fro": float(np.linalg.norm(m)), "spectral": float(sv[0]),
                "l1": float(np.abs(m).sum()), "absmax": float(np.abs(m).max()),
                "eff_rank": eff, "stable_rank": stab, "numer_rank": numer})
            flat = m.ravel()
            WSAMPLE[key][p].append(flat[rng.integers(0, flat.size, WSAMPLE_PER_LP)])
            if key != REF:
                a = baseW[p].ravel()
                DRIFT[key][p].append(
                    float(a @ flat / (np.linalg.norm(a) * np.linalg.norm(flat) + 1e-8)))
        DOWN_NORM[key][L] = np.linalg.norm(W["down"], axis=0)      # per-neuron out-norm
        pick = rng.choice(INTERMEDIATE, NSAMP_PER_LAYER, replace=False)
        _nf[key].append(W["gate"][pick].astype(np.float32))        # gate rows = in-dirs
        _nl[key].append(np.full(NSAMP_PER_LAYER, L))
        if key != REF:
            del W
    del baseW
    if (L + 1) % 8 == 0:
        print(f"  processed layer {L+1:2d}/{N_LAYERS}  ({time.time()-t0:5.0f}s)")

for key in MODEL_KEYS:
    NEUR[key] = (np.concatenate(_nf[key]), np.concatenate(_nl[key]))
    for p in PROJS:
        WSAMPLE[key][p] = np.concatenate(WSAMPLE[key][p])
del _nf, _nl; gc.collect()
norm_df = pd.DataFrame(STATREC)
print(f"done in {time.time()-t0:.0f}s | STATREC rows = {len(norm_df)}")

## A1b · Per-layer weight drift from base

Cosine similarity between the base model's projection and each SinLlama stage's,
per layer — how far continual pretraining moved each MLP matrix (1 = unchanged).
A direct 4-model training-progression signal, per gate/up/down.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, p in zip(axes, PROJS):
    for key in MODEL_KEYS:
        if key == REF:
            continue
        dr = np.array(DRIFT[key][p])
        ax.plot(dr, marker="o", ms=3, label=key)
        SUMMARY[key][f"mean_drift_{p}"] = float(dr.mean())
    ax.set_title(f"{p}_proj cosine to base"); ax.set_xlabel("layer")
axes[0].set_ylabel("cos(base, variant)"); axes[0].legend(fontsize=8)
savefig("A1b_weight_drift.png")
print("Mean drift from base (lower = moved more):")
for key in MODEL_KEYS:
    if key == REF:
        continue
    print("  " + key + ": " +
          ", ".join(f"{p}={np.mean(DRIFT[key][p]):.4f}" for p in PROJS))

## A2 · §2 Basic weight statistics

Element-wise mean/std/min/max/median/percentiles per projection, the weight-value
distribution, and outlier neurons (by down-column norm).

In [ ]:
display(norm_df.groupby(["model", "proj"])[
    ["mean", "std", "median", "p1", "p99", "absmax"]].mean().round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, p in zip(axes, PROJS):
    for key in MODEL_KEYS:
        ax.hist(WSAMPLE[key][p], bins=160, density=True, alpha=0.5, label=key)
    ax.set_title(f"{p}_proj weight values (sampled)")
    ax.set_xlabel("weight value"); ax.set_yscale("log")
axes[0].set_ylabel("density (log)"); axes[0].legend(fontsize=8)
savefig("A2_weight_dist.png")

# Outlier neurons: down-column norm > mean + 4 sigma, per model (pooled layers).
for key in MODEL_KEYS:
    dn = DOWN_NORM[key].ravel()
    thr = dn.mean() + 4 * dn.std()
    out = np.where(dn > thr)[0]
    print(f"{key}: {len(out)} outlier neurons (down-norm > mean+4sigma); "
          f"max down-norm = {dn.max():.3f} at (L{out[np.argmax(dn[out])]//INTERMEDIATE}, "
          f"n{out[np.argmax(dn[out])]%INTERMEDIATE})" if len(out) else
          f"{key}: no >4sigma outlier neurons; max down-norm = {dn.max():.3f}")

## A3 · §3 Matrix-norm analysis

Frobenius, spectral (top singular value) and entry-wise L1 norm per
`(layer, proj)` — which layers apply the strongest transformation, and does
magnitude grow with depth?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric, title in zip(
        axes, ("fro", "spectral", "l1"),
        ("Frobenius ||W||_F", "spectral ||W||_2", "entry-wise L1")):
    for key in MODEL_KEYS:
        sub = norm_df[norm_df.model == key]
        by_layer = sub.groupby("layer")[metric].mean()
        ax.plot(by_layer.index, by_layer.values, marker="o", ms=3, label=key)
    ax.set_title(f"{title} vs layer (mean over gate/up/down)"); ax.set_xlabel("layer")
axes[0].set_ylabel("norm"); axes[0].legend(fontsize=8)
savefig("A3_norms_by_layer.png")

# gate/up/down Frobenius side-by-side for the reference model.
piv = norm_df[norm_df.model == REF].pivot_table(index="layer", columns="proj", values="fro")
print(f"{REF}: strongest-Frobenius layers per projection:")
for p in PROJS:
    print(f"  {p}: L{int(piv[p].idxmax())} (||W||_F={piv[p].max():.1f}), "
          f"weakest L{int(piv[p].idxmin())} ({piv[p].min():.1f})")

## A4 · §4 Singular-value / spectral analysis

Singular spectra (computed once in §1), cumulative energy, and effective /
stable rank vs depth. Are MLP projections low-rank, and does rank change with
depth or differ between gate/up/down?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
Lmid = N_LAYERS // 2
for key in MODEL_KEYS:
    sv = SVSPEC[(key, Lmid, "gate")]
    ax[0].semilogy(sv, alpha=0.8, label=key)
    ax[1].plot(np.cumsum(sv ** 2) / (sv ** 2).sum(), alpha=0.8, label=key)
ax[0].set_title(f"gate_proj singular spectrum (L{Lmid}, log)")
ax[0].set_xlabel("index"); ax[0].set_ylabel("singular value"); ax[0].legend(fontsize=8)
ax[1].set_title(f"gate_proj cumulative energy (L{Lmid})")
ax[1].set_xlabel("index"); ax[1].set_ylabel("fraction of energy"); ax[1].legend(fontsize=8)
for key in MODEL_KEYS:
    ax[2].plot(SVEFF[key]["gate"], marker="o", ms=3, label=key)
ax[2].set_title("gate_proj effective rank vs depth")
ax[2].set_xlabel("layer"); ax[2].set_ylabel(f"effective rank (of {HIDDEN})"); ax[2].legend(fontsize=8)
savefig("A4_svd_effrank.png")

for key in MODEL_KEYS:
    for p in PROJS:
        SUMMARY[key][f"{p}_effrank"] = float(np.mean(SVEFF[key][p]))
    print(f"{key}: eff-rank mean  " +
          ", ".join(f"{p}={np.mean(SVEFF[key][p]):.0f}" for p in PROJS) +
          f"  | stable-rank gate={np.mean(SVSTAB[key]['gate']):.1f}")

## A9 · §9 Neuron similarity (redundant / duplicate neurons)

Each neuron's **gate row** is its input-reading direction. Pairwise cosine among
a per-layer sample finds duplicate / redundant neurons; the heatmap shows a
mid-layer block for the reference model.

In [ ]:
def top_duplicate_pairs(feats, layer_of, key, per_layer=256, k=5):
    "Highest cosine off-diagonal gate-row pairs within a few probe layers."
    found = []
    for L in SPEC_LAYERS:
        idx = np.where(layer_of == L)[0]
        v = feats[idx]; v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
        S = v @ v.T; np.fill_diagonal(S, -1)
        i, j = np.unravel_index(S.argmax(), S.shape)
        found.append((L, int(i), int(j), float(S[i, j])))
    return found

feats, layer_of = NEUR[REF]
idxm = np.where(layer_of == N_LAYERS // 2)[0][:200]
v = feats[idxm]; v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
plt.figure(figsize=(6.8, 5.8))
sns.heatmap(v @ v.T, cmap="magma", square=True, cbar_kws={"label": "cosine (gate row)"})
plt.title(f"Neuron gate-row similarity — {REF} L{N_LAYERS//2} (200 sampled)")
plt.xlabel("neuron (sampled)"); plt.ylabel("neuron (sampled)")
savefig("A9_neuron_similarity.png")

for key in MODEL_KEYS:
    fk, lk = NEUR[key]
    dup = top_duplicate_pairs(fk, lk, key)
    SUMMARY[key]["max_neuron_cos"] = max(c for *_, c in dup)
    print(f"{key}: most-similar sampled gate pairs -> "
          + ", ".join(f"(L{L}:{i}~{j},{c:.2f})" for L, i, j, c in dup))

## A10 · §10 + §11 Neuron dimensionality reduction & clustering (weight-space)

Sampled neurons (gate rows) projected with PCA / UMAP and coloured by layer,
then K-Means grouped — do neurons organise by depth, and where do the SinLlama
branches differ from base?

In [ ]:
NEURFEAT = {}
for key in MODEL_KEYS:
    feats, layer_of = NEUR[key]
    Xs = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)
    p50 = PCA(n_components=50, svd_solver="randomized", random_state=SEED).fit_transform(Xs)
    pca2 = p50[:, :2]
    um = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(p50)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for coords, a, name in [(pca2, ax[0], "PCA"), (um, ax[1], "UMAP")]:
        sc = a.scatter(coords[:, 0], coords[:, 1], c=layer_of, cmap="viridis", s=8, alpha=0.6)
        a.set_title(f"{key} — {name} of sampled neurons (colour = layer)")
    fig.colorbar(sc, ax=ax[1], label="layer depth")
    savefig(f"A10_neuron_pca_umap_{key}.png")
    km = KMeans(n_clusters=8, n_init=10, random_state=SEED).fit(p50)
    NEURFEAT[key] = (p50, layer_of, km.labels_)
    print(f"{key}: neuron clusters concentrate at depths -> "
          + str({c: int(np.median(layer_of[km.labels_ == c])) for c in range(8)}))

In [ ]:
# Ward dendrogram of sampled neurons for the reference model.
p50, layer_of, _ = NEURFEAT[REF]
sel = rng.choice(len(p50), size=800, replace=False)
Z = linkage(p50[sel], method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z, no_labels=True, color_threshold=0.7 * Z[:, 2].max())
plt.title(f"Ward hierarchical clustering of sampled neurons — {REF}")
plt.ylabel("merge distance")
savefig("A11_neuron_dendrogram.png")

In [ ]:
# Free Part-A weight-derived arrays not needed by Part B (keep DOWN_NORM, SUMMARY).
del NEUR, NEURFEAT, WSAMPLE, SVSPEC
gc.collect()
print("Released Part-A neuron samples. DOWN_NORM kept for §7 importance.")

# PART B — Dynamic / activation-space analysis

Real forward passes with hooks on `gate_proj` / `up_proj` (pre-activations),
`down_proj` (its input = the post-SwiGLU activation), the `mlp` module (residual
delta) and `self_attn` (for the MLP-vs-attention comparison). Auto-selects GPU
if it fits, else CPU. Each full model is loaded once and freed.

## B0 · Device guard, model loader & sample inputs

In [ ]:
def pick_device(param_billion=8.03, dtype_bytes=2, overhead=1.4):
    if torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        if free > param_billion * 1e9 * dtype_bytes * overhead:
            return "cuda"
    return "cpu"

DEVICE = pick_device()
DTYPE = torch.bfloat16
MAX_LEN = 48 if DEVICE == "cpu" else 96
print(f"Part B device = {DEVICE} | dtype = {DTYPE} | MAX_LEN = {MAX_LEN}")
if DEVICE == "cpu":
    print("NOTE: 8B forwards on CPU are slow. On the MI300X this runs on GPU.")

DYN_MODELS = list(MODEL_KEYS)          # trim to e.g. [REF, "SinLlama_Instruct"] on small boxes

SAMPLE_TEXTS = [
    "The quick brown fox jumps over the lazy dog near the river bank.",
    "In 2024, sales rose by 15% to $3.2 million, up from 2.8 last year.",
    "She said, \"Come here!\" and then walked away without another word.",
    "ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි දිවයිනකි. එහි අගනුවර කොළඹ නගරයයි.",
]

def load_full_model(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=DTYPE, low_cpu_mem_usage=True)
    return model.to(DEVICE).eval(), tok

ACT_THRESH = 1e-2                        # |post-SwiGLU| below this counts as inactive
DYN = {}

# --- shared Part-B activation helpers (used by B1's per-model loop AND the
#     reference-only cells B12/B18) ------------------------------------------
STOP = {"the","a","an","of","to","in","and","is","was","that","it","for","on","with"}
def token_type(tok, tid):
    s = tok.decode([int(tid)]).strip()
    if tid in set(tok.all_special_ids): return "special"
    if s == "": return "whitespace"
    if re.search(r"[඀-෿]", s): return "sinhala"
    if s.isdigit(): return "number"
    if all(not c.isalnum() for c in s): return "punctuation"
    if s.lower() in STOP: return "stopword"
    return "english"

def capture_post(model, tok, text):
    "Per-token post-SwiGLU activations [L, S, I] + token ids (down_proj pre-hooks)."
    store = {}; handles = []
    def mk(L):
        def f(mod, args): store[L] = args[0].detach()[0].float().cpu().numpy()
        return f
    for L in range(N_LAYERS):
        handles.append(model.model.layers[L].mlp.down_proj.register_forward_pre_hook(mk(L)))
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        model(**enc)
    for h in handles:
        h.remove()
    ids = enc["input_ids"][0].cpu().numpy()
    return np.stack([store[L] for L in range(N_LAYERS)]), ids     # [L, S, I], [S]

CONCEPTS = {
    "english":  ["The weather in London is cold and rainy today.",
                 "He read an interesting book about ancient history."],
    "sinhala":  ["අද කාලගුණය සිසිල් සහ වැසි සහිතයි.",
                 "ඔහු පුරාණ ඉතිහාසය ගැන පොතක් කියෙව්වා."],
    "numbers":  ["1 2 3 4 5 6 7 8 9 10 11 12 13 14 15.",
                 "The total was 4827 plus 1936 equals 6763."],
    "code":     ["def add(a, b):\n    return a + b",
                 "for i in range(10): print(i * i)"],
    "dates":    ["On January 5th, 2021 the meeting was rescheduled.",
                 "Between 1990 and 2005 the population doubled."],
}
def concept_profile(model, tok, prompts):
    "Mean |post-SwiGLU| per neuron over a concept's prompt tokens -> [L, I]."
    acc = np.zeros((N_LAYERS, INTERMEDIATE)); n = 0
    for p in prompts:
        post, ids = capture_post(model, tok, p)
        acc += np.abs(post).sum(1); n += post.shape[1]
    return acc / n

concept_counts = {}                      # model -> {concept: #selective neurons}
CONCEPT_PROFS_REF = None                  # (profiles, global) for the reference model

## B1 · §5 + §6 Neuron activation & sparsity collection

One hooked pass over the sample texts per model, accumulating **per-neuron**
running statistics (mean |activation|, variance, activation frequency, max) and
the co-statistics needed for gate-vs-up analysis, plus **per-layer** residual
bookkeeping (MLP vs attention contribution, representation change). Raw
activations are aggregated then discarded — only the compact summaries survive.

In [ ]:
def collect_mlp_stats(model, tok, texts):
    L_, I_, H_ = N_LAYERS, INTERMEDIATE, HIDDEN
    s_abs = np.zeros((L_, I_)); s_sq = np.zeros((L_, I_)); s_raw = np.zeros((L_, I_))
    s_max = np.zeros((L_, I_)); active = np.zeros((L_, I_))
    sg = np.zeros((L_, I_)); su = np.zeros((L_, I_))
    sgg = np.zeros((L_, I_)); suu = np.zeros((L_, I_)); sgu = np.zeros((L_, I_))
    both = np.zeros((L_, I_))
    mlp_n = np.zeros(L_); attn_n = np.zeros(L_); resid_n = np.zeros(L_); cos_io = np.zeros(L_)
    n_tok = 0; n_seq = 0

    store = {}; handles = []
    def mk_post(L):
        def f(mod, args): store[(L, "post")] = args[0].detach()[0].float().cpu().numpy()
        return f
    def mk_out(L, tag):
        def f(mod, inp, out): store[(L, tag)] = out.detach()[0].float().cpu().numpy()
        return f
    def mk_mlp(L):
        def f(mod, inp, out): store[(L, "mlp")] = out.detach()[0].float().cpu().numpy()
        return f
    def mk_attn(L):
        def f(mod, inp, out):
            o = out[0] if isinstance(out, tuple) else out
            store[(L, "attn")] = o.detach()[0].float().cpu().numpy()
        return f
    def mk_blk(L):
        def f(mod, args): store[(L, "blk")] = args[0].detach()[0].float().cpu().numpy()
        return f
    for L in range(L_):
        lay = model.model.layers[L]
        handles += [lay.register_forward_pre_hook(mk_blk(L)),
                    lay.self_attn.register_forward_hook(mk_attn(L)),
                    lay.mlp.gate_proj.register_forward_hook(mk_out(L, "gate")),
                    lay.mlp.up_proj.register_forward_hook(mk_out(L, "up")),
                    lay.mlp.down_proj.register_forward_pre_hook(mk_post(L)),
                    lay.mlp.register_forward_hook(mk_mlp(L))]
    try:
        for t in texts:
            store.clear()
            enc = tok(t, return_tensors="pt", truncation=True,
                      max_length=MAX_LEN).to(model.device)
            with torch.no_grad():
                model(**enc)
            S = enc["input_ids"].shape[1]; n_tok += S; n_seq += 1
            for L in range(L_):
                post = store[(L, "post")]; gate = store[(L, "gate")]; up = store[(L, "up")]
                s_abs[L] += np.abs(post).sum(0); s_sq[L] += (post ** 2).sum(0)
                s_raw[L] += post.sum(0); s_max[L] = np.maximum(s_max[L], np.abs(post).max(0))
                a = np.abs(post) > ACT_THRESH; active[L] += a.sum(0)
                sg[L] += gate.sum(0); su[L] += up.sum(0)
                sgg[L] += (gate ** 2).sum(0); suu[L] += (up ** 2).sum(0); sgu[L] += (gate * up).sum(0)
                both[L] += ((np.abs(gate) > ACT_THRESH) & (np.abs(up) > ACT_THRESH)).sum(0)
                blk = store[(L, "blk")]; at = store[(L, "attn")]; mo = store[(L, "mlp")]
                r_before = blk + at
                mlp_n[L] += np.linalg.norm(mo, axis=1).mean()
                attn_n[L] += np.linalg.norm(at, axis=1).mean()
                resid_n[L] += np.linalg.norm(r_before, axis=1).mean()
                r_after = r_before + mo
                cs = (r_before * r_after).sum(1) / (
                    np.linalg.norm(r_before, axis=1) * np.linalg.norm(r_after, axis=1) + 1e-8)
                cos_io[L] += cs.mean()
    finally:
        for h in handles:
            h.remove()
    mean_abs = s_abs / n_tok
    var = s_sq / n_tok - (s_raw / n_tok) ** 2
    mg, mu = sg / n_tok, su / n_tok
    cov = sgu / n_tok - mg * mu
    corr = cov / (np.sqrt(np.clip(sgg / n_tok - mg ** 2, 1e-12, None)) *
                  np.sqrt(np.clip(suu / n_tok - mu ** 2, 1e-12, None)) + 1e-12)
    return {"mean_abs": mean_abs, "var": var, "freq": active / n_tok,
            "max": s_max, "gate_up_corr": corr, "gate_up_overlap": both / n_tok,
            "mlp_norm": mlp_n / n_seq, "attn_norm": attn_n / n_seq,
            "resid_norm": resid_n / n_seq, "cos_inout": cos_io / n_seq}

# Load each model ONCE, extract EVERY per-model array (activation stats +
# concept profiles used by B13), then FREE non-reference models immediately.
# Only the reference model stays resident, for the reference-only cells below.
# Holding all four 8B models at once exhausts GPU memory (especially with the
# three notebooks running in parallel); that previously aborted this loop after
# the first model, leaving DYN with one model and the 4-model plots blank.
for key in DYN_MODELS:
    print(f">>> loading {key} on {DEVICE} ...")
    model, model_tok = load_full_model(MODEL_PATHS[key])
    DYN[key] = collect_mlp_stats(model, model_tok, SAMPLE_TEXTS)
    profs = {c: concept_profile(model, model_tok, ps) for c, ps in CONCEPTS.items()}
    glob = np.mean(list(profs.values()), axis=0)
    cc = {}
    for c, pr in profs.items():
        sel = pr - glob
        cc[c] = int((sel > sel.mean() + 3 * sel.std()).sum())
    concept_counts[key] = cc
    if key == REF:
        CONCEPT_PROFS_REF = (profs, glob)
    d = DYN[key]
    dead = (d["freq"] < 0.01).sum(); spars = 1 - d["freq"].mean()
    SUMMARY[key]["mean_sparsity"] = float(spars)
    SUMMARY[key]["dead_neurons"] = int(dead)
    print(f"{key}: mean activation sparsity = {spars:.3f} | "
          f"near-dead neurons (freq<1%) = {dead} / {N_LAYERS*INTERMEDIATE}")
    if key == REF:
        DYN[key]["_model"] = model; DYN[key]["_tok"] = model_tok
    else:
        del model, model_tok; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# Activation-statistics overview + per-layer sparsity.
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for key in DYN_MODELS:
    ax[0].hist(DYN[key]["mean_abs"].ravel(), bins=160, density=True, alpha=0.5, label=key)
ax[0].set_title("Per-neuron mean |post-SwiGLU| activation")
ax[0].set_xlabel("mean |activation|"); ax[0].set_ylabel("density"); ax[0].set_yscale("log")
ax[0].legend(fontsize=8)
for key in DYN_MODELS:
    ax[1].plot(1 - DYN[key]["freq"].mean(1), marker="o", ms=3, label=key)
ax[1].set_title("Activation sparsity vs layer (fraction inactive)")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("sparsity"); ax[1].legend(fontsize=8)
savefig("B5_activation_stats.png")

## B6 · §6 Neuron sparsity — layer-wise & dead-neuron map

Fraction of neurons that essentially never fire (activation frequency < 1%) per
layer, plus the near-dead count per model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for key in DYN_MODELS:
    dead_frac = (DYN[key]["freq"] < 0.01).mean(1)
    ax[0].plot(dead_frac, marker="o", ms=3, label=key)
ax[0].set_title("Fraction of near-dead neurons (freq<1%) vs layer")
ax[0].set_xlabel("layer"); ax[0].set_ylabel("dead fraction"); ax[0].legend(fontsize=8)
for key in DYN_MODELS:
    ax[1].plot(DYN[key]["freq"].mean(1), marker="o", ms=3, label=key)
ax[1].set_title("Mean activation frequency vs layer")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("mean firing frequency"); ax[1].legend(fontsize=8)
savefig("B6_sparsity.png")

## B7 · §7 Neuron importance

Importance = mean |post-SwiGLU activation| × ||down-column|| — a neuron's typical
contribution magnitude to the residual stream (activation × output weight).
Ranks neurons and shows how concentrated the computation is.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for key in DYN_MODELS:
    imp = DYN[key]["mean_abs"] * DOWN_NORM[key]
    DYN[key]["importance"] = imp
    ax[0].plot(imp.sum(1), marker="o", ms=3, label=key)       # total importance / layer
    flat = np.sort(imp.ravel())[::-1]
    ax[1].plot(np.cumsum(flat) / flat.sum(), label=key)       # concentration curve
    top = np.dstack(np.unravel_index(np.argsort(imp.ravel())[::-1][:5], imp.shape))[0]
    frac10 = flat[:int(0.1 * flat.size)].sum() / flat.sum()
    SUMMARY[key]["imp_top10pct_share"] = float(frac10)
    print(f"{key}: top-10% neurons hold {frac10*100:.1f}% of importance | "
          f"top neurons (L,n): " + ", ".join(f"(L{L},{n})" for L, n in top))
ax[0].set_title("Total neuron importance vs layer"); ax[0].set_xlabel("layer")
ax[0].set_ylabel("sum importance"); ax[0].legend(fontsize=8)
ax[1].set_title("Cumulative importance (sorted neurons)")
ax[1].set_xlabel("neuron rank fraction"); ax[1].set_ylabel("cumulative share"); ax[1].legend(fontsize=8)
savefig("B7_importance.png")

## B8 · §8 Gate vs Up projection (SwiGLU gating)

The SwiGLU output is `SiLU(gate) ⊙ up`. Per-neuron correlation between the gate
and up pre-activations, and their co-activation overlap, characterise how the
two branches interact.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for key in DYN_MODELS:
    corr = DYN[key]["gate_up_corr"].ravel()
    corr = corr[np.isfinite(corr)]
    ax[0].hist(corr, bins=120, density=True, alpha=0.5, label=f"{key} (mean={corr.mean():+.2f})")
    ax[1].plot(DYN[key]["gate_up_overlap"].mean(1), marker="o", ms=3, label=key)
    SUMMARY[key]["gate_up_corr"] = float(corr.mean())
ax[0].set_title("Per-neuron gate–up activation correlation")
ax[0].set_xlabel("corr(gate, up)"); ax[0].set_ylabel("density"); ax[0].axvline(0, color="k", ls="--", lw=1)
ax[0].legend(fontsize=8)
ax[1].set_title("Gate & up co-activation overlap vs layer")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("fraction both active"); ax[1].legend(fontsize=8)
savefig("B8_gate_vs_up.png")

## B12 · §12 Token-conditioned MLP activation (reference model)

Which token *types* drive the MLP hardest? Mean post-SwiGLU activation energy
per token category, plus the neurons most selective for each category.

In [ ]:
model, tok = DYN[REF]["_model"], DYN[REF]["_tok"]
cat_energy = {}; cat_meanvec = {}
for t in SAMPLE_TEXTS:
    post, ids = capture_post(model, tok, t)                       # [L, S, I]
    types = [token_type(tok, i) for i in ids]
    for s, ty in enumerate(types):
        cat_energy.setdefault(ty, []).append(float(np.abs(post[:, s, :]).mean()))
        cat_meanvec.setdefault(ty, []).append(post[N_LAYERS // 2, s, :])
order = sorted(cat_energy, key=lambda k: -np.mean(cat_energy[k]))
plt.figure(figsize=(8, 4.2))
sns.barplot(x=order, y=[np.mean(cat_energy[k]) for k in order])
plt.ylabel("mean |post-SwiGLU| activation"); plt.xticks(rotation=25)
plt.title(f"MLP activation energy by token type — {REF}")
savefig("B12_token_conditioned.png")
for ty in order:
    mv = np.abs(np.mean(cat_meanvec[ty], 0))
    print(f"  {ty:11s} energy={np.mean(cat_energy[ty]):.3f} | "
          f"top L{N_LAYERS//2} neurons: {list(np.argsort(-mv)[:5])}")

## B13 · §13 Concept-neuron discovery

Run short concept prompts (English, Sinhala, numbers, code, dates), collect
per-neuron activation, and score selectivity as `concept_mean − global_mean`.
Counts strongly-selective neurons per concept per model — a probe of whether the
SinLlama branches grew Sinhala-selective feed-forward neurons.

In [ ]:
# concept_counts / CONCEPT_PROFS_REF were filled in B1 while each model was
# live (the models are freed by now), so this cell only tabulates and plots.
cc = pd.DataFrame(concept_counts).T[list(CONCEPTS)]
display(cc)

profs, glob = CONCEPT_PROFS_REF
print(f"{REF}: top concept-selective neurons (L,n):")
for c, pr in profs.items():
    sel = (pr - glob).ravel()
    top = np.unravel_index(np.argsort(sel)[::-1][:3], pr.shape)
    print(f"  {c:9s}: " + ", ".join(f"(L{L},{n})" for L, n in zip(*top)))

cc.plot(kind="bar", figsize=(9, 4.2))
plt.ylabel("# strongly-selective neurons (>3 sigma)")
plt.title("Concept-selective MLP neurons per model"); plt.xticks(rotation=15)
plt.legend(fontsize=8, ncol=3); savefig("B13_concept_neurons.png")

## B14 · §14 Layer-wise MLP evolution (all dynamic models)

Activation entropy (how spread firing is across neurons), neuron specialisation
(coefficient of variation of per-neuron activation), and mean firing frequency —
tracked across depth for every model.

In [ ]:
def layer_entropy(mean_abs):
    p = mean_abs / (mean_abs.sum(1, keepdims=True) + 1e-12)
    return -(p * np.log(p + 1e-12)).sum(1)          # per layer, over neurons

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for key in DYN_MODELS:
    ma = DYN[key]["mean_abs"]
    ent = layer_entropy(ma)
    spec = ma.std(1) / (ma.mean(1) + 1e-12)
    ax[0].plot(ent, marker="o", ms=3, label=key)
    ax[1].plot(spec, marker="o", ms=3, label=key)
    ax[2].plot(DYN[key]["freq"].mean(1), marker="o", ms=3, label=key)
    SUMMARY[key]["mean_act_entropy"] = float(ent.mean())
ax[0].set_title("activation entropy vs depth"); ax[0].set_ylabel("entropy (nats)")
ax[1].set_title("neuron specialisation (CoV) vs depth")
ax[2].set_title("mean firing frequency vs depth")
for a in ax:
    a.set_xlabel("layer"); a.legend(fontsize=8)
savefig("B14_layerwise_evolution.png")

## B15 · §15 MLP output-space analysis

How much each layer's MLP rotates the residual — cosine between the residual
before and after the MLP add — and the relative magnitude of the MLP delta.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for key in DYN_MODELS:
    ax[0].plot(DYN[key]["cos_inout"], marker="o", ms=3, label=key)
    ax[1].plot(DYN[key]["mlp_norm"] / (DYN[key]["resid_norm"] + 1e-8),
               marker="o", ms=3, label=key)
ax[0].set_title("cos(residual before, after MLP) vs layer")
ax[0].set_xlabel("layer"); ax[0].set_ylabel("cosine"); ax[0].legend(fontsize=8)
ax[1].set_title("relative MLP contribution ||mlp|| / ||residual|| vs layer")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("ratio"); ax[1].legend(fontsize=8)
savefig("B15_output_space.png")

## B17 · §17 MLP vs attention contribution

Both sub-blocks write to the residual stream; compare their delta magnitudes per
layer to see which component dominates at each depth.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for key in DYN_MODELS:
    ax[0].plot(DYN[key]["mlp_norm"], marker="o", ms=3, label=f"{key} MLP")
    ax[0].plot(DYN[key]["attn_norm"], marker="x", ms=3, ls="--", label=f"{key} attn")
    ax[1].plot(DYN[key]["mlp_norm"] / (DYN[key]["attn_norm"] + 1e-8),
               marker="o", ms=3, label=key)
    SUMMARY[key]["mlp_attn_ratio"] = float(np.mean(
        DYN[key]["mlp_norm"] / (DYN[key]["attn_norm"] + 1e-8)))
ax[0].set_title("residual delta magnitude: MLP vs attention")
ax[0].set_xlabel("layer"); ax[0].set_ylabel("mean ||delta||"); ax[0].legend(fontsize=7, ncol=2)
ax[1].set_title("MLP / attention contribution ratio vs layer")
ax[1].set_xlabel("layer"); ax[1].set_ylabel("||mlp|| / ||attn||"); ax[1].legend(fontsize=8)
savefig("B17_mlp_vs_attn.png")

## B16 · §16 Causal analysis — neuron ablation (reference model)

Zero each layer's **top-k most-important** neurons (from §7) at the `down_proj`
input and measure the LM-loss increase, versus ablating a random-k baseline —
does the importance score identify causally-relevant neurons? `N_LAYERS` forwards
per condition; trivial on the MI300X.

In [ ]:
ABLATION_TEXT = "The capital of France is Paris and the capital of Japan is Tokyo."
ABL_K = 128

@torch.no_grad()
def ablate_importance(model, tok, importance, k=ABL_K):
    ids = tok(ABLATION_TEXT, return_tensors="pt").to(model.device)["input_ids"]
    base = model(ids, labels=ids).loss.item()
    top_d, rnd_d = [], []
    for L in range(N_LAYERS):
        top = np.argsort(-importance[L])[:k]
        rnd = rng.choice(INTERMEDIATE, size=k, replace=False)
        for idxs, bucket in ((top, top_d), (rnd, rnd_d)):
            down = model.model.layers[L].mlp.down_proj
            def hook(mod, args, idxs=idxs):
                x = args[0].clone(); x[..., idxs] = 0
                return (x,) + args[1:]
            h = down.register_forward_pre_hook(hook)
            bucket.append(model(ids, labels=ids).loss.item() - base)
            h.remove()
    return base, np.array(top_d), np.array(rnd_d)

base, top_d, rnd_d = ablate_importance(
    DYN[REF]["_model"], DYN[REF]["_tok"], DYN[REF]["importance"])
plt.figure(figsize=(10, 4.2))
plt.plot(top_d, marker="o", ms=3, label=f"top-{ABL_K} important")
plt.plot(rnd_d, marker="x", ms=3, ls="--", label=f"random-{ABL_K}")
plt.title(f"Δ LM-loss when ablating {ABL_K} neurons / layer — {REF} (base={base:.3f})")
plt.xlabel("layer"); plt.ylabel("loss increase"); plt.legend(fontsize=8)
savefig("B16_ablation.png")
print(f"{REF}: mean Δloss top={top_d.mean():.4f} vs random={rnd_d.mean():.4f} "
      f"| most causal layers: {list(np.argsort(-top_d)[:5])}")

## B18 · §18 Frequency-based analysis (optional)

Enable by dropping a `{token_id: count}` JSON at `weights_analysis/token_freq.json`;
correlates token frequency with the mean MLP activation energy each token elicits
in the reference model. Skips cleanly if the file is absent.

In [ ]:
FREQ_PATH = os.path.join(FIG_DIR, "..", "token_freq.json")
if os.path.exists(FREQ_PATH):
    freq_map = {int(k): v for k, v in json.load(open(FREQ_PATH)).items()}
    model, tok = DYN[REF]["_model"], DYN[REF]["_tok"]
    tok_freq, tok_energy = [], []
    for t in SAMPLE_TEXTS:
        post, ids = capture_post(model, tok, t)              # [L, S, I]
        e = np.abs(post).mean((0, 2))                        # per-token mean energy [S]
        for s, tid in enumerate(ids):
            if freq_map.get(int(tid), 0) > 0:
                tok_freq.append(freq_map[int(tid)]); tok_energy.append(float(e[s]))
    lf, en = np.log1p(np.array(tok_freq)), np.array(tok_energy)
    plt.figure(figsize=(7, 5))
    plt.scatter(lf, en, s=10, alpha=0.4)
    plt.xlabel("log(1 + token frequency)"); plt.ylabel("mean MLP activation energy")
    plt.title(f"Token frequency vs MLP activation energy — {REF}")
    savefig("B18_freq_vs_activation.png")
    print(f"{REF}: corr(log-freq, MLP energy) = {np.corrcoef(lf, en)[0,1]:.3f}")
else:
    print(f"No frequency file at {FREQ_PATH} -> skipping section 18 (optional).")

In [ ]:
# Free the full models.
for key in DYN_MODELS:
    DYN[key].pop("_model", None); DYN[key].pop("_tok", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released full models.")

# PART C — §20 Synthesis: research questions

In [ ]:
summary_df = pd.DataFrame(SUMMARY).T
display(summary_df.round(3))

def g(key, m, d=float("nan")):
    return SUMMARY.get(key, {}).get(m, d)

print("\n================ ANSWERS ================")
for key in MODEL_KEYS:
    print(f"[{key}]  gate eff-rank={g(key,'gate_effrank'):.0f}/{HIDDEN}, "
          f"sparsity={g(key,'mean_sparsity'):.3f}, dead={g(key,'dead_neurons'):.0f}, "
          f"imp top-10%={g(key,'imp_top10pct_share'):.2f}, "
          f"gate-up corr={g(key,'gate_up_corr'):+.2f}, "
          f"mlp/attn={g(key,'mlp_attn_ratio'):.2f}, "
          f"drift gate={g(key,'mean_drift_gate'):.3f}")
print("\n• Are MLP matrices low-rank?      -> A4 effective/stable rank vs 4096.")
print("• Are representations sparse?     -> B5/B6 sparsity & dead-neuron counts.")
print("• Is computation concentrated?    -> B7 top-10% importance share + B16 ablation.")
print("• Concept / language neurons?     -> B13 selective-neuron counts (Sinhala vs base).")
print("• Reasoning vs retrieval by depth -> B15/B17 MLP-vs-attention contribution.")
print("• Cleanest training-stage signal  -> A1b weight drift (base -> v01 -> {cpt, instruct}).")

### Reading the 4-model comparison

* **Part A weight-space** separates the branches structurally: per-layer drift
  (A1b, vs base), norms (A3), effective/stable rank (A4) and neuron organisation
  (A9-A11) show *where* `Llama-3 → v01 → {cpt, instruct}` reshaped the
  feed-forward maps — compare the cpt and instruct siblings against their shared
  parent v01.
* **Part B activation-space** depends on the input language: the Sinhala sample
  exercises the SinLlama models' adapted neurons. Watch sparsity / dead neurons
  (B5-B6), importance concentration (B7), gate–up gating (B8), and especially the
  concept-neuron counts (B13) — did continual pretraining grow Sinhala-selective
  MLP neurons?
* **MLP vs attention (B15/B17)** locates where each model does residual "writing"
  work by depth; **ablation (B16)** validates that the importance score marks
  causally-relevant neurons.
* On the MI300X, add longer / more `SAMPLE_TEXTS`, raise `NSAMP_PER_LAYER`, and
  widen the concept prompt sets for publication-grade Part-B results.